<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [1]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [ ]:
import numpy as np

def vectorizar(texto, vocabulario):
    palabras = texto.split()
    return np.array([1 if palabra in palabras else 0 for palabra in vocabulario])

# Vectorizamos todos los correos del corpus
X = np.array([vectorizar(texto, vocabulario) for _, texto in corpus])
y = np.array([etiqueta for etiqueta, _ in corpus])

for (etiqueta, texto), vector in zip(corpus, X):
    print(f"{etiqueta:7s} | {texto:40s} -> {vector}")

# Verificación manual de dos ejemplos
print()
print("gana dinero gratis ->", vectorizar("gana dinero gratis", vocabulario), "(esperado [1 1 0 0 0])")
print("reunion de proyecto ->", vectorizar("reunion de proyecto", vocabulario), "(esperado [0 0 0 1 1])")

spam    | gana dinero gratis                       -> [1 1 0 0 0]
spam    | dinero gratis ahora                      -> [1 1 0 0 0]
spam    | premio dinero gratis                     -> [1 1 1 0 0]
spam    | gana premio ahora                        -> [0 0 1 0 0]
spam    | en la reunion habra dinero gratis        -> [1 1 0 0 1]
normal  | daremos un premio despues de la reunion  -> [0 0 1 0 1]
normal  | reunion de proyecto                      -> [0 0 0 1 1]
normal  | proyecto para mañana                     -> [0 0 0 1 0]
normal  | reunion mañana                           -> [0 0 0 0 1]
normal  | informe del proyecto                     -> [0 0 0 1 0]
normal  | informe del proyecto                     -> [0 0 0 1 0]

gana dinero gratis -> [1 1 0 0 0] (esperado [1 1 0 0 0])
reunion de proyecto -> [0 0 0 1 1] (esperado [0 0 0 1 1])


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [3]:
# Calculamos P(Y=spam) y P(Y=normal)
n_total = len(corpus)
n_spam = np.sum(y == "spam")
n_normal = np.sum(y == "normal")

P_spam = n_spam / n_total
P_normal = n_normal / n_total

print(f"P(Y=spam)   = {n_spam}/{n_total} = {P_spam:.4f}")
print(f"P(Y=normal) = {n_normal}/{n_total} = {P_normal:.4f}")

P(Y=spam)   = 5/11 = 0.4545
P(Y=normal) = 6/11 = 0.5455


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [4]:
import pandas as pd

X_spam = X[y == "spam"]
X_normal = X[y == "normal"]

# P(X_i = 1 | Y = y) sin suavizado: N_iy / N_y
P_i_given_spam = X_spam.mean(axis=0)
P_i_given_normal = X_normal.mean(axis=0)

tabla_sin_suavizado = pd.DataFrame({
    "P(Xi=1|spam)": P_i_given_spam,
    "P(Xi=1|normal)": P_i_given_normal
}, index=vocabulario)

tabla_sin_suavizado

,P(Xi=1|spam),P(Xi=1|normal)
dinero,0.8,0.000000
gratis,0.8,0.000000
premio,0.4,0.166667
proyecto,0.0,0.666667
reunion,0.2,0.500000


**Respuesta:** si una palabra nunca aparece en una clase (por ejemplo "proyecto" en spam), su probabilidad estimada es $P(X_i=1\mid Y=y)=0$. Como Naive Bayes multiplica probabilidades condicionales, un solo cero anula por completo la verosimilitud $P(x\mid Y=y)$ de cualquier correo futuro que contenga esa palabra, sin importar la evidencia de las demás palabras. Esto motiva el suavizado de Laplace de la Parte 4.

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [5]:
# P(X_i = 1 | Y = y) con suavizado de Laplace: (N_iy + 1) / (N_y + 2)
N_iy_spam = X_spam.sum(axis=0)
N_iy_normal = X_normal.sum(axis=0)

P_i_given_spam_laplace = (N_iy_spam + 1) / (n_spam + 2)
P_i_given_normal_laplace = (N_iy_normal + 1) / (n_normal + 2)

tabla_laplace = pd.DataFrame({
    "P(Xi=1|spam)": P_i_given_spam_laplace,
    "P(Xi=1|normal)": P_i_given_normal_laplace
}, index=vocabulario)

tabla_laplace

,P(Xi=1|spam),P(Xi=1|normal)
dinero,0.714286,0.125
gratis,0.714286,0.125
premio,0.428571,0.250
proyecto,0.142857,0.625
reunion,0.285714,0.500


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [6]:
def likelihood(x, P_i_given_y):
    """P(x | Y=y) = prod_i P(Xi=xi | Y=y), usando probabilidades con Laplace."""
    p_uno = P_i_given_y          # P(Xi=1|Y=y)
    p_cero = 1 - P_i_given_y     # P(Xi=0|Y=y)
    probs = np.where(x == 1, p_uno, p_cero)
    return np.prod(probs)

x_nuevo = vectorizar("dinero gratis", vocabulario)
print("Vector x =", x_nuevo)

# Verosimilitudes P(x|Y=spam) y P(x|Y=normal)
L_spam = likelihood(x_nuevo, P_i_given_spam_laplace)
L_normal = likelihood(x_nuevo, P_i_given_normal_laplace)
print(f"P(x|Y=spam)   = {L_spam:.6f}")
print(f"P(x|Y=normal) = {L_normal:.6f}")

# Scores conjuntos S_y = P(Y=y) * P(x|Y=y)
S_spam = P_spam * L_spam
S_normal = P_normal * L_normal
print(f"\nS_spam   = P(Y=spam)*P(x|spam)     = {S_spam:.6f}")
print(f"S_normal = P(Y=normal)*P(x|normal) = {S_normal:.6f}")

# Posterior normalizado P(Y=y|x)
Z = S_spam + S_normal
P_spam_given_x = S_spam / Z
P_normal_given_x = S_normal / Z
print(f"\nP(Y=spam|x)   = {P_spam_given_x:.4f}")
print(f"P(Y=normal|x) = {P_normal_given_x:.4f}")

clase_predicha = "spam" if P_spam_given_x > P_normal_given_x else "normal"
print(f"\nClase predicha para 'dinero gratis': {clase_predicha}")

Vector x = [1 1 0 0 0]
P(x|Y=spam)   = 0.178497
P(x|Y=normal) = 0.002197

S_spam   = P(Y=spam)*P(x|spam)     = 0.081135
S_normal = P(Y=normal)*P(x|normal) = 0.001199

P(Y=spam|x)   = 0.9854
P(Y=normal|x) = 0.0146

Clase predicha para 'dinero gratis': spam


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?